In [57]:
# ============================================================
# 044_meeting_presentation_generator_svg
# ============================================================
#
# Overview
# ----------------
# Generates meeting presentations via an SVG-first workflow optimized for Microsoft PowerPoint.
# The agenda is converted into a slide plan, Gemini generates PowerPoint-compatible SVG (1920x1080)
# with real <text> elements, SVGs are sanitized + validated, then delivered as:
# (a) saved per-slide SVG assets and (b) a PPTX assembled from those assets.
# Two assembly options exist: python-pptx (SVG→PNG via cairosvg) and an optional AppleScript path
# that inserts SVG files directly into PowerPoint.
#
# Inputs / Outputs
# ----------------
# Inputs:
#   - Meeting agenda (dict / widgets; optional meeting_agenda.json)
#   - Style + layout constants (canvas, margins, fonts, colors)
#   - Gemini API credentials (GEMINI_API_KEY)
#
# Outputs:
#   - Per-slide SVG files saved under a run-timestamped output folder
#   - svg_manifest.json (saved file list + sizes)
#   - PPTX file:
#       - python-pptx path: SVG converted to PNG and embedded full-slide
#       - optional AppleScript path: SVG inserted into PowerPoint slides
#   - Reports:
#       - validation results (in-memory + optional JSON artifacts)
#       - final_report.json + delivery_package (PPTX, SVGs, reports, README)
#
# Structure
# ----------------
# Cell 01: Environment setup and imports
# Cell 02: Configuration and constants (canvas/layout/fonts/colors + output paths)
# Cell 03: Gemini API client initialization (google.genai)
# Cell 04: Slide planning + widgets UI + run-timestamped output directories
# Cell 05: SVG prompt builder (PowerPoint-optimized, transparency/external-ref constraints)
# Cell 06: SVG sanitize + validation layer (structure, <text> required, prohibited tags/attrs, background rect first)
# Cell 07: Gemini SVG generation with validate-aware retry + namespace normalization
# Cell 08: Save validated SVGs to disk + verification + svg_manifest.json
# Cell 09: PPTX assembly (SVG→PNG via cairosvg + python-pptx) + pptx_metadata.json
# Cell 10: Final reporting + delivery package creation/validation
# Cell 11 (Optional): PowerPoint AppleScript assembly inserting SVG files directly
#
# Notes
# ----------------
# - SVG generation is strict for PowerPoint stability:
#     - Real <text> elements required (no text-to-path)
#     - No <image>, base64, scripts, foreignObject, use/a, defs/gradients/filters/masks/clipPath
#     - No opacity / alpha (opacity, fill-opacity, stroke-opacity, #RRGGBBAA, rgba/hsla, url(...))
#     - First child must be a full-canvas solid background <rect> with numeric size and HEX fill
# - Namespaces are normalized to http:// URIs for PowerPoint compatibility.
# - Validation failures trigger retry with explicit error feedback to Gemini.
# - The python-pptx assembly path embeds rasterized PNGs (not shape-editable in PPTX);
#   the optional AppleScript path is provided when direct SVG insertion/edit workflow is needed.
# - Output paths are run-timestamped to avoid overwriting prior runs.
# - Delivery package bundles PPTX + SVG assets + reports + README for handoff.


In [48]:
# ============================================================
# Cell 01 — Environment setup and imports
# ============================================================
# Overview:
#   Initializes the runtime environment, loads environment variables,
#   sets LLM configuration, and imports required libraries.
#
# Inputs / Outputs:
#   Inputs:
#     - env.txt (environment variables, e.g., API keys)
#   Outputs:
#     - Loaded environment variables
#     - LLM configuration variables
#     - Imported modules ready for use
#
# Notes:
#   - Ensure env.txt exists and required packages are installed.
#   - Safe to re-run if needed.
#


# --- Mandatory env loading ---
from dotenv import load_dotenv
load_dotenv('env.txt')

# --- Runtime LLM configuration (given / assumed) ---
llm_provider = 'OpenAI'
llm_model = 'gpt-4o-mini'
llm_temperature = 0.0

# --- Standard library imports ---
import os
import json
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

# --- Third-party imports ---
from google import genai
from pptx import Presentation
from pptx.util import Inches, Pt
import xml.etree.ElementTree as ET
from xml.dom import minidom

# --- Display and logging ---
from IPython.display import display, SVG, HTML
import warnings
warnings.filterwarnings('ignore')

print("✓ Environment loaded")
print(f"✓ LLM Configuration: {llm_provider} / {llm_model} / temp={llm_temperature}")
print("✓ All imports successful")


✓ Environment loaded
✓ LLM Configuration: OpenAI / gpt-4o-mini / temp=0.0
✓ All imports successful


In [49]:
# ============================================================
# Cell 02 — Configuration and constants
# ============================================================
# Overview:
#   Defines global configuration for SVG generation, layout,
#   styling, output paths, and Gemini API settings.
#
# Inputs / Outputs:
#   Inputs:
#     - Optional meeting_agenda.json (agenda source file)
#   Outputs:
#     - Global constants for canvas, fonts, colors, layout
#     - Output directories and timestamped file paths
#
# Notes:
#   - SVG canvas is fixed at 1920x1080 (Full HD).
#   - Output filenames include millisecond timestamps.
#   - Validation rules restrict unsafe SVG elements.
#   - Directories are created automatically if missing.
#


# 追加：ファイル名用タイムスタンプ
from datetime import datetime
from zoneinfo import ZoneInfo

def _ts_ms(tz="Asia/Tokyo"):
    # 例: 20260212_205725_123
    return datetime.now(ZoneInfo(tz)).strftime("%Y%m%d_%H%M%S_%f")[:19]  # %f(μs)をms相当に短縮


GEMINI_MODEL_NAME = "models/gemini-2.5-pro"
# --- SVG Canvas Configuration ---
SVG_WIDTH = 1920
SVG_HEIGHT = 1080
SVG_VIEWBOX = f"0 0 {SVG_WIDTH} {SVG_HEIGHT}"

# --- Font Configuration ---
FONT_FAMILY = "Meiryo"
FONT_SIZE_TITLE = 64
FONT_SIZE_HEADING = 48
FONT_SIZE_BODY = 32
FONT_SIZE_CAPTION = 24

# --- Color Palette ---
COLOR_PRIMARY = "#2C3E50"
COLOR_SECONDARY = "#3498DB"
COLOR_ACCENT = "#E74C3C"
COLOR_TEXT_DARK = "#2C3E50"
COLOR_TEXT_LIGHT = "#FFFFFF"
COLOR_BACKGROUND = "#FFFFFF"
COLOR_BACKGROUND_ALT = "#ECF0F1"

# --- Layout Configuration ---
MARGIN_TOP = 100
MARGIN_BOTTOM = 100
MARGIN_LEFT = 120
MARGIN_RIGHT = 120
CONTENT_WIDTH = SVG_WIDTH - MARGIN_LEFT - MARGIN_RIGHT
CONTENT_HEIGHT = SVG_HEIGHT - MARGIN_TOP - MARGIN_BOTTOM

# --- Output Paths ---
OUTPUT_DIR = Path("output")
SVG_DIR = OUTPUT_DIR / "svg_slides"
PPTX_OUTPUT = OUTPUT_DIR / f"meeting_presentation_{_ts_ms()}.pptx"
VALIDATION_REPORT = OUTPUT_DIR / f"validation_report_{_ts_ms()}.json"

# Create output directories
OUTPUT_DIR.mkdir(exist_ok=True)
SVG_DIR.mkdir(exist_ok=True)

# --- Prohibited SVG Elements (validation) ---
PROHIBITED_ELEMENTS = [
    'image',
    'script',
    'foreignObject',
    'use',  # external references
    'a',    # links
]

# --- Required SVG Elements (validation) ---
REQUIRED_ELEMENTS = [
    'svg',
    'text',
]

# --- SVG Generation Rules ---
SVG_RULES = {
    "canvas": f"{SVG_WIDTH}x{SVG_HEIGHT}",
    "viewBox": SVG_VIEWBOX,
    "text_elements_required": True,
    "no_path_text": True,
    "no_images": True,
    "no_base64": True,
    "no_external_refs": True,
    "inline_styles_only": True,
    "font_family": FONT_FAMILY,
}

# --- Gemini API Configuration ---
GEMINI_MODEL = "gemini-2.5-pro"
GEMINI_TEMPERATURE = 0.7
GEMINI_MAX_RETRIES = 3

# --- Sample Meeting Agenda Path ---
AGENDA_JSON_PATH = Path("meeting_agenda.json")

print("✓ Configuration loaded")
print(f"✓ SVG Canvas: {SVG_WIDTH}x{SVG_HEIGHT}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print(f"✓ SVG directory: {SVG_DIR}")
print(f"✓ Font family: {FONT_FAMILY}")
print(f"✓ Gemini model: {GEMINI_MODEL}")


✓ Configuration loaded
✓ SVG Canvas: 1920x1080
✓ Output directory: output
✓ SVG directory: output/svg_slides
✓ Font family: Meiryo
✓ Gemini model: gemini-2.5-pro


In [50]:
# ============================================================
# Cell 03 — Gemini API client initialization (google.genai版)
# ============================================================
# Overview:
#   Initializes the Gemini API client using credentials
#   loaded from environment variables.
#
# Inputs / Outputs:
#   Inputs:
#     - GEMINI_API_KEY (from env.txt or environment)
#   Outputs:
#     - Initialized `gemini_client` instance
#
# Notes:
#   - Raises an error if GEMINI_API_KEY is not set.
#   - Uses google.genai SDK.
#   - Relies on previously defined GEMINI_MODEL,
#     GEMINI_TEMPERATURE, and GEMINI_MAX_RETRIES.
#


from google import genai

# --- Retrieve Gemini API key from environment ---
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in environment. Please set it in env.txt")

# --- Initialize Gemini client ---
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("✓ Gemini API key loaded")
print("✓ Gemini Client initialized (google.genai)")
print(f"✓ Model: {GEMINI_MODEL}")
print(f"✓ Temperature: {GEMINI_TEMPERATURE}")
print(f"✓ Max retries: {GEMINI_MAX_RETRIES}")


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✓ Gemini API key loaded
✓ Gemini Client initialized (google.genai)
✓ Model: gemini-2.5-pro
✓ Temperature: 0.7
✓ Max retries: 3


In [51]:
# ============================================================
# Cell 04 — Slide content structure planning (Widget版 + 日時付き出力パス)
# ============================================================
# Overview:
#   Defines slide content data structures and a planner that converts an agenda dict
#   into a list of slide specs. Also builds an interactive ipywidgets UI to edit the
#   agenda and generate `slide_plan_global`. Output paths are set per-run with a timestamp.
#
# Inputs / Outputs:
#   Inputs:
#     - meeting_agenda (optional; uses DEFAULT_AGENDA if missing)
#     - OUTPUT_DIR (optional; falls back to "output")
#   Outputs:
#     - RUN_TS, OUTPUT_DIR, SVG_DIR, PPTX_OUTPUT (timestamped per run)
#     - meeting_agenda (updated from widgets on Build)
#     - slide_plan_global (list[SlideContent]) created on Build
#     - Displayed widgets UI + console summaries
#
# Notes:
#   - Uses ipywidgets; intended for notebook execution.
#   - Output directories are created automatically and avoid overwriting via RUN_TS.
#   - The "Build" button updates globals; downstream cells should read slide_plan_global.
#   - Topic add/remove is handled dynamically via widget callbacks.
#


from dataclasses import dataclass
from typing import List, Optional, Dict, Any
from enum import Enum
from pathlib import Path
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# -----------------------------
# (1) Slide definitions (既存を維持)
# -----------------------------
class SlideType(Enum):
    TITLE = "title"
    TOPIC = "topic"
    SUMMARY = "summary"
    ACTION_ITEMS = "action_items"


@dataclass
class SlideContent:
    slide_number: int
    slide_type: SlideType
    title: str
    subtitle: Optional[str] = None
    body_points: List[str] = None
    metadata: Dict[str, Any] = None

    def __post_init__(self):
        if self.body_points is None:
            self.body_points = []
        if self.metadata is None:
            self.metadata = {}


def plan_slide_structure(agenda: Dict) -> List[SlideContent]:
    slides = []
    slide_number = 1

    # Title slide
    slides.append(SlideContent(
        slide_number=slide_number,
        slide_type=SlideType.TITLE,
        title=agenda.get("title", "会議資料"),
        subtitle=agenda.get("date", ""),
        metadata={"attendees": agenda.get("attendees", [])},
    ))
    slide_number += 1

    # Topic slides
    for topic in agenda.get("topics", []):
        slides.append(SlideContent(
            slide_number=slide_number,
            slide_type=SlideType.TOPIC,
            title=topic.get("title", "トピック"),
            subtitle=None,
            body_points=topic.get("points", []),
            metadata={
                "duration": topic.get("duration", ""),
                "presenter": topic.get("presenter", ""),
            },
        ))
        slide_number += 1

    # Action items slide (optional)
    if agenda.get("action_items"):
        slides.append(SlideContent(
            slide_number=slide_number,
            slide_type=SlideType.ACTION_ITEMS,
            title="アクションアイテム",
            subtitle=None,
            body_points=agenda.get("action_items", []),
            metadata={},
        ))

    return slides


def format_slide_summary(slide: SlideContent) -> str:
    parts = [f"Slide {slide.slide_number}: {slide.slide_type.value}", f"  Title: {slide.title}"]
    if slide.subtitle:
        parts.append(f"  Subtitle: {slide.subtitle}")
    if slide.body_points:
        parts.append(f"  Body points: {len(slide.body_points)}")
        for i, p in enumerate(slide.body_points[:3], 1):
            parts.append(f"    {i}. {p}")
        if len(slide.body_points) > 3:
            parts.append(f"    ... and {len(slide.body_points) - 3} more")
    if slide.metadata:
        for k, v in slide.metadata.items():
            if v:
                parts.append(f"  {k}: {v}")
    return "\n".join(parts)


# -----------------------------
# (2) 実行ごとに「日時付き」の出力先へ（上書き防止）
# -----------------------------
RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

try:
    _base_output = Path(OUTPUT_DIR)  # 既存があれば親を使う
except Exception:
    _base_output = Path("output")

OUTPUT_DIR = _base_output / f"run_{RUN_TS}"
SVG_DIR = OUTPUT_DIR / "svgs"
PPTX_OUTPUT = OUTPUT_DIR / f"meeting_presentation_{RUN_TS}.pptx"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SVG_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Run timestamp: {RUN_TS}")
print(f"✓ OUTPUT_DIR = {OUTPUT_DIR}")
print(f"✓ SVG_DIR    = {SVG_DIR}")
print(f"✓ PPTX_OUTPUT= {PPTX_OUTPUT}")


# -----------------------------
# (3) Agenda → Widgets
#    ★ meeting_agenda が無くても止まらず進む（ここが修正点）
# -----------------------------
DEFAULT_AGENDA = {
    "title": "会議資料",
    "date": "",
    "attendees": [],
    "topics": [
        {"title": "トピック1", "presenter": "", "duration": "", "points": []},
    ],
    "action_items": [],
}

agenda_src = meeting_agenda if "meeting_agenda" in globals() and isinstance(meeting_agenda, dict) else DEFAULT_AGENDA

w_title = widgets.Text(value=str(agenda_src.get("title", "会議資料")), description="タイトル", layout=widgets.Layout(width="80%"))
w_date = widgets.Text(value=str(agenda_src.get("date", "")), description="日付", layout=widgets.Layout(width="50%"))
w_attendees = widgets.Textarea(
    value="、".join(agenda_src.get("attendees", []) or []),
    description="出席者",
    layout=widgets.Layout(width="80%", height="70px")
)

w_action_items = widgets.Textarea(
    value="\n".join(agenda_src.get("action_items", []) or []),
    description="Action",
    layout=widgets.Layout(width="80%", height="120px")
)

topics_box = widgets.VBox([])
topic_editors = []  # each item: dict of widgets

def _make_topic_editor(topic: Dict[str, Any], idx: int):
    wt = widgets.Text(value=str(topic.get("title", f"トピック{idx+1}")), description="題名", layout=widgets.Layout(width="80%"))
    wp = widgets.Text(value=str(topic.get("presenter", "")), description="担当", layout=widgets.Layout(width="50%"))
    wd = widgets.Text(value=str(topic.get("duration", "")), description="時間", layout=widgets.Layout(width="30%"))
    wpoints = widgets.Textarea(
        value="\n".join(topic.get("points", []) or []),
        description="箇条書き",
        layout=widgets.Layout(width="80%", height="140px")
    )

    remove_btn = widgets.Button(description="このトピックを削除", button_style="danger", icon="trash")
    container = widgets.VBox([
        widgets.HBox([wt]),
        widgets.HBox([wp, wd]),
        wpoints,
        remove_btn
    ])

    editor = {"title": wt, "presenter": wp, "duration": wd, "points": wpoints, "container": container}
    def _on_remove(_):
        nonlocal editor
        if editor in topic_editors:
            topic_editors.remove(editor)
        _refresh_topics_ui()

    remove_btn.on_click(_on_remove)
    return editor

def _refresh_topics_ui():
    topics_box.children = [ed["container"] for ed in topic_editors]

# init topics
for i, t in enumerate(agenda_src.get("topics", []) or []):
    topic_editors.append(_make_topic_editor(t, i))
_refresh_topics_ui()

add_topic_btn = widgets.Button(description="トピック追加", button_style="info", icon="plus")
def _on_add_topic(_):
    topic_editors.append(_make_topic_editor({"title":"", "presenter":"", "duration":"", "points":[]}, len(topic_editors)))
    _refresh_topics_ui()
add_topic_btn.on_click(_on_add_topic)

build_btn = widgets.Button(description="Build slide_plan_global", button_style="success", icon="play")
out = widgets.Output()

def _parse_lines(text: str) -> List[str]:
    lines = []
    for ln in (text or "").splitlines():
        s = ln.strip()
        if s:
            lines.append(s)
    return lines

def _build_agenda_from_widgets() -> Dict[str, Any]:
    topics = []
    for ed in topic_editors:
        topics.append({
            "title": ed["title"].value.strip() or "トピック",
            "presenter": ed["presenter"].value.strip(),
            "duration": ed["duration"].value.strip(),
            "points": _parse_lines(ed["points"].value),
        })

    attendees = []
    raw_att = (w_attendees.value or "").strip()
    if raw_att:
        if "、" in raw_att:
            attendees = [x.strip() for x in raw_att.split("、") if x.strip()]
        else:
            attendees = _parse_lines(raw_att)

    agenda = {
        "title": w_title.value.strip() or "会議資料",
        "date": w_date.value.strip(),
        "attendees": attendees,
        "topics": topics,
        "action_items": _parse_lines(w_action_items.value),
    }
    return agenda

def _on_build(_):
    global meeting_agenda, slide_plan_global

    with out:
        clear_output()

        meeting_agenda = _build_agenda_from_widgets()
        slide_plan_global = plan_slide_structure(meeting_agenda)

        print("✓ meeting_agenda updated from widgets")
        print(f"✓ Planned {len(slide_plan_global)} slides\n")
        print("="*60)
        for s in slide_plan_global:
            print(format_slide_summary(s))
            print("-"*60)

        print("\n✓ slide_plan_global is ready (next: SVG generation)")

build_btn.on_click(_on_build)

ui = widgets.VBox([
    widgets.HTML("<h3>Meeting → Slide Plan Builder</h3>"),
    widgets.HBox([w_title]),
    widgets.HBox([w_date]),
    w_attendees,
    widgets.HTML("<h4>Topics</h4>"),
    add_topic_btn,
    topics_box,
    widgets.HTML("<h4>Action Items</h4>"),
    w_action_items,
    widgets.HBox([build_btn]),
    out
])

display(ui)


✓ Run timestamp: 20260213_091919
✓ OUTPUT_DIR = output/run_20260213_091919
✓ SVG_DIR    = output/run_20260213_091919/svgs
✓ PPTX_OUTPUT= output/run_20260213_091919/meeting_presentation_20260213_091919.pptx


In [58]:
# ============================================================
# Cell 05 — SVG prompt builder (PowerPoint完全最適版 / transparency対策)
# ============================================================
# Overview:
#   Builds strict LLM prompts to generate PowerPoint-friendly, editable SVG slides and
#   provides helpers to (a) create a retry prompt after validation failures and
#   (b) extract a clean SVG block from an LLM response.
#
# Inputs / Outputs:
#   Inputs:
#     - SlideContent (title/subtitle/body_points/metadata + slide_type)
#     - SlideType (enum, defined in another cell)
#     - failed_svg, validation_errors (for retry prompt)
#     - LLM raw response text (for SVG extraction)
#   Outputs:
#     - Prompt strings for SVG generation / regeneration
#     - Extracted SVG string (XML declaration + <svg>...</svg>) or None
#
# Notes:
#   - Enforces PowerPoint compatibility: numeric canvas size, HEX colors, no opacity,
#     no external refs, no images/scripts/styles, and real <text> elements.
#   - Layout differs by SlideType via get_slide_layout_config().
#   - `extract_svg_from_response` tolerates fenced code blocks and prioritizes XML+SVG.
#


import re
from typing import Optional, List

from typing import Dict, Any

# --- Canvas ---
SVG_WIDTH = 1920
SVG_HEIGHT = 1080
SVG_VIEWBOX = f"0 0 {SVG_WIDTH} {SVG_HEIGHT}"

# --- Margins ---
MARGIN_TOP = 120
MARGIN_BOTTOM = 120
MARGIN_LEFT = 120
MARGIN_RIGHT = 120

# --- Typography ---
FONT_FAMILY = "Hiragino Sans, Hiragino Kaku Gothic ProN, Yu Gothic, Meiryo, sans-serif"
FONT_SIZE_TITLE = 64
FONT_SIZE_HEADING = 48
FONT_SIZE_BODY = 32

# --- Colors (HEX only; PPT互換重視) ---
COLOR_PRIMARY = "#0B3D91"
COLOR_SECONDARY = "#1F2937"
COLOR_BACKGROUND = "#FFFFFF"
COLOR_BACKGROUND_ALT = "#F3F4F6"
COLOR_TEXT_DARK = "#111827"
COLOR_TEXT_LIGHT = "#FFFFFF"

def get_slide_layout_config(slide_type) -> Dict[str, Any]:
    # SlideType Enum を前提（Cell 06 か Widgetセルで定義されている想定）
    layouts = {
        SlideType.TITLE: {
            "title_y": SVG_HEIGHT // 2 - 80,
            "title_size": FONT_SIZE_TITLE,
            "subtitle_y": SVG_HEIGHT // 2 + 20,
            "subtitle_size": FONT_SIZE_HEADING,
            "background_color": COLOR_PRIMARY,
            "text_color": COLOR_TEXT_LIGHT,
        },
        SlideType.TOPIC: {
            "title_y": MARGIN_TOP + 80,
            "title_size": FONT_SIZE_HEADING,
            "body_start_y": MARGIN_TOP + 200,
            "body_line_height": 60,
            "body_size": FONT_SIZE_BODY,
            "background_color": COLOR_BACKGROUND,
            "text_color": COLOR_TEXT_DARK,
        },
        SlideType.SUMMARY: {
            "title_y": MARGIN_TOP + 80,
            "title_size": FONT_SIZE_HEADING,
            "body_start_y": MARGIN_TOP + 200,
            "body_line_height": 60,
            "body_size": FONT_SIZE_BODY,
            "background_color": COLOR_BACKGROUND_ALT,
            "text_color": COLOR_TEXT_DARK,
        },
        SlideType.ACTION_ITEMS: {
            "title_y": MARGIN_TOP + 80,
            "title_size": FONT_SIZE_HEADING,
            "body_start_y": MARGIN_TOP + 200,
            "body_line_height": 60,
            "body_size": FONT_SIZE_BODY,
            "background_color": COLOR_SECONDARY,
            "text_color": COLOR_TEXT_LIGHT,
        },
    }
    return layouts.get(slide_type, layouts[SlideType.TOPIC])

print("✓ Layout constants ready")
def build_svg_prompt(slide: SlideContent) -> str:
    """
    PowerPointに貼り付け→「図形に変換」しやすいSVGを出させるための厳格プロンプト（透明化対策込み）
    - 背景rectは % ではなく数値(width=SVG_WIDTH, height=SVG_HEIGHT)で固定（PPT互換）
    - opacity/alpha禁止
    - style属性より「属性指定」を優先（PPTが崩れにくい）
    - コメント禁止
    """

    layout = get_slide_layout_config(slide.slide_type)
    font_family = FONT_FAMILY

    prompt_parts = [
        "You are an expert SVG generator for Microsoft PowerPoint import.",
        "Generate ONE slide as pure SVG that remains editable after PowerPoint 'Convert to Shape'.",
        "",
        "CRITICAL OUTPUT RULES (MUST FOLLOW):",
        "1) Output ONLY raw SVG code. No markdown fences. No explanations. No extra text.",
        "2) The first characters MUST be exactly: <?xml version=\"1.0\" encoding=\"UTF-8\"?>",
        f"3) <svg> MUST include: width=\"{SVG_WIDTH}\" height=\"{SVG_HEIGHT}\" viewBox=\"{SVG_VIEWBOX}\"",
        '3.1) <svg> MUST include exactly: xmlns="http://www.w3.org/2000/svg" (include the "http://").'
        "   - Add preserveAspectRatio=\"none\"",
        "   - Add xmlns=\"http://www.w3.org/2000/svg\"",
        "4) Place a full-canvas BACKGROUND rectangle FIRST with numeric size (NOT %):",
        f"   <rect x=\"0\" y=\"0\" width=\"{SVG_WIDTH}\" height=\"{SVG_HEIGHT}\" fill=\"#RRGGBB\"/>",
        "   (SOLID fill only. No opacity. No style=.)",
        "5) All text MUST be real <text> elements. NEVER convert text to <path>.",
        f"6) Font-family MUST include: {font_family}",
        "7) ABSOLUTELY PROHIBITED:",
        "   <image>, <script>, <foreignObject>, <use>, <a>, <style>, base64, external URLs, url(...),",
        "   patterns, masks, filters, clipPath, linearGradient, radialGradient",
        "8) DO NOT use opacity (no opacity attribute, no fill-opacity, no stroke-opacity). Use solid colors only.",
        "9) Prefer SVG presentation attributes over style= (fill=, stroke=, font-size=, font-family=, font-weight=...).",
        "10) No comments at all (do not output <!-- ... -->).",
        "",
        "CAN USE ONLY THESE ELEMENTS:",
        "- <svg>, <rect>, <line>, <circle>, <ellipse>, <polygon>, <path>, <text>",
        "  (Use <path> only for simple decorative shapes, never for text.)",
        "",
        "- Background MUST be:"
        '<rect x="0" y="0" width="1920" height="1080" fill="#XXXXXX"/>'
        "  (NO %, NO opacity, NO style attribute)"
        "- DO NOT use preserveAspectRatio"
        "- DO NOT use percentage width/height"
        "- DO NOT use multiple font fallbacks"
        '- Use font-family="Meiryo"'
        "POWERPOINT COMPATIBILITY NOTES (MUST FOLLOW):",
        "- Use only HEX colors like #RRGGBB (no rgba(), no hsl(), no named colors).",
        "- Use integer coordinates whenever possible.",
        "- Avoid group-level styles; set attributes directly on each element.",
        "- Avoid % units (x/y/width/height). Use numeric pixels only.",
        "- Avoid CSS shorthand; avoid style attribute if possible.",
        "- Avoid fill=\"none\" on important shapes. (For lines, set stroke and omit fill.)",
        "",
        "LAYOUT CONSTRAINTS:",
        f"- Canvas: {SVG_WIDTH}x{SVG_HEIGHT}px",
        f"- Margins: top={MARGIN_TOP}px, bottom={MARGIN_BOTTOM}px, left={MARGIN_LEFT}px, right={MARGIN_RIGHT}px",
        f"- Background color: {layout['background_color']} (must be converted to #RRGGBB)",
        f"- Default text color: {layout['text_color']} (must be converted to #RRGGBB)",
        "",
        "IMPORTANT: Convert any colors you use into strict #RRGGBB format.",
        "",
    ]

    cx = SVG_WIDTH // 2

    if slide.slide_type == SlideType.TITLE:
        prompt_parts.extend([
            "SLIDE TYPE: Title Slide",
            "RENDER THESE CONTENTS:",
            f"- Main title centered: \"{slide.title}\"",
            f"  - x={cx}, y={layout['title_y']}, font-size={layout['title_size']}, font-weight=700",
        ])

        if slide.subtitle:
            prompt_parts.extend([
                f"- Subtitle centered: \"{slide.subtitle}\"",
                f"  - x={cx}, y={layout['subtitle_y']}, font-size={layout['subtitle_size']}, font-weight=400",
            ])

        if slide.metadata.get("attendees"):
            attendees_str = "、".join(slide.metadata["attendees"])
            prompt_parts.extend([
                f"- Attendees (bottom-right): \"{attendees_str}\"",
                f"  - x={SVG_WIDTH - MARGIN_RIGHT}, y={SVG_HEIGHT - MARGIN_BOTTOM}, font-size=22, text-anchor=end",
            ])

    else:
        prompt_parts.extend([
            f"SLIDE TYPE: {slide.slide_type.value.replace('_',' ').title()}",
            "RENDER THESE CONTENTS:",
            f"- Title left-aligned: \"{slide.title}\"",
            f"  - x={MARGIN_LEFT}, y={layout['title_y']}, font-size={layout['title_size']}, font-weight=700",
        ])

        presenter = slide.metadata.get("presenter")
        duration = slide.metadata.get("duration")
        if presenter:
            prompt_parts.append(
                f"- Top-right small text: \"{presenter}\" at x={SVG_WIDTH - MARGIN_RIGHT}, y={MARGIN_TOP + 40}, "
                "font-size=24, text-anchor=end"
            )
        if duration:
            prompt_parts.append(
                f"- Top-right small text (below presenter): \"{duration}\" at x={SVG_WIDTH - MARGIN_RIGHT}, y={MARGIN_TOP + 75}, "
                "font-size=22, text-anchor=end"
            )

        if slide.body_points:
            prompt_parts.extend([
                "- Body bullet points:",
                f"  - bullet x={MARGIN_LEFT + 24}, text x={MARGIN_LEFT + 48}, start y={layout['body_start_y']}",
                f"  - font-size={layout['body_size']}, line-height={layout['body_line_height']}",
                "- Use bullet char '●' for each line, as its own <text> element.",
                "",
                "BULLET POINTS:",
            ])
            for i, p in enumerate(slide.body_points, 1):
                prompt_parts.append(f"{i}. {p}")

    prompt_parts.extend([
        "",
        "MANDATORY SVG STRUCTURE (FOLLOW THIS ORDER):",
        "A) XML declaration",
        "B) <svg width height viewBox preserveAspectRatio xmlns ...>",
        f"C) Background FIRST: <rect x=\"0\" y=\"0\" width=\"{SVG_WIDTH}\" height=\"{SVG_HEIGHT}\" fill=\"#RRGGBB\"/>",
        "D) Then any decorative shapes (rect/line/circle/ellipse/polygon/path) with solid colors only",
        "E) Then all <text> elements (each has fill, font-family, font-size, text-anchor as needed)",
        "F) Close </svg>",
        "",
        "REMEMBER: Output ONLY the raw SVG. No markdown. No comments. No opacity. No external references.",
    ])

    return "\n".join(prompt_parts)


def build_validation_prompt(slide: SlideContent, failed_svg: str, validation_errors: List[str]) -> str:
    base = build_svg_prompt(slide)

    retry_parts = [
        base,
        "",
        "============================================================",
        "PREVIOUS ATTEMPT FAILED VALIDATION. FIX AND REGENERATE.",
        "============================================================",
        "VALIDATION ERRORS:",
    ]
    for i, e in enumerate(validation_errors, 1):
        retry_parts.append(f"{i}. {e}")

    retry_parts.extend([
        "",
        "FIX CHECKLIST (MUST):",
        "- Output must start with the XML declaration exactly.",
        "- Must contain a complete <svg ...>...</svg> root.",
        "- Must contain at least one <text> element (real text).",
        "- MUST NOT include: http://, https://, url(...), opacity, comments, or prohibited tags.",
        f"- Background must be FIRST and must be numeric: width=\"{SVG_WIDTH}\" height=\"{SVG_HEIGHT}\" (NOT %).",
        "- Use attributes instead of style= wherever possible.",
        "",
        "Now output the corrected SVG ONLY.",
    ])

    return "\n".join(retry_parts)


def extract_svg_from_response(text: str) -> Optional[str]:
    """
    PowerPoint向けに「XML宣言 + <svg>...</svg>」を優先して抜く。
    - fenced code blockがあってもOK
    - XML宣言が含まれていれば一緒に返す
    """
    if not text:
        return None

    s = text.strip()

    # 1) fenced
    m = re.search(r"```(?:svg|xml)?\s*([\s\S]*?)\s*```", s, flags=re.IGNORECASE)
    if m:
        s = m.group(1).strip()

    # 2) Capture optional xml decl + svg
    m = re.search(
        r"(\<\?xml[\s\S]*?\?\>\s*)?(<svg\b[\s\S]*?</svg>)",
        s,
        flags=re.IGNORECASE
    )
    if not m:
        if re.search(r"<svg\b", s, flags=re.IGNORECASE):
            return None
        return None

    xml_decl = m.group(1) or ""
    svg_block = m.group(2)
    out = (xml_decl + svg_block).strip()

    if "<svg" not in out.lower() or "</svg>" not in out.lower():
        return None

    return out


✓ Layout constants ready


In [59]:
# ============================================================
# Cell 06 — SVG validation layer (PowerPoint完全最適) + sanitize
# ============================================================
# Overview:
#   Validates and sanitizes generated SVGs to ensure PowerPoint-friendly compliance.
#   Checks XML/SVG structure, requires <text> elements, blocks unsafe/incompatible tags
#   and attributes (e.g., opacity, external refs), and enforces a solid full-canvas
#   background <rect> as the first element. Provides batch validation over all slides.
#
# Inputs / Outputs:
#   Inputs:
#     - SVG strings (single SVG via validate_svg_structure / batch via validate_all_generated_svgs)
#     - svg_results: Dict[int, (svg_content, generation_success, gen_errors)]
#     - Global constants (SVG_WIDTH/HEIGHT, FONT_FAMILY) defined in prior cells
#   Outputs:
#     - Sanitized SVG string (from sanitize_svg_remove_external_refs)
#     - (is_valid, [SVGValidationError...]) per SVG
#     - Batch validation summary printed to stdout
#     - Dict[int, (is_valid, [SVGValidationError...])] from validate_all_generated_svgs
#
# Notes:
#   - Designed to avoid common PowerPoint import issues (transparency/filters/URLs).
#   - Sanitizer removes <image> and external hrefs without breaking xmlns declarations.
#   - Background rule is strict: first child must be a full-canvas solid rect (#RRGGBB, no opacity).
#   - Errors vs warnings are separated; validity is based on absence of "error" severity items.
#


from typing import List, Tuple, Dict, Optional
import xml.etree.ElementTree as ET
import re

SVG_NS = "http://www.w3.org/2000/svg"
XLINK_NS = "http://www.w3.org/1999/xlink"

# PowerPointで壊れやすい/透明化しやすい/外部依存になりやすい要素は明示的に禁止
PROHIBITED_ELEMENTS = [
    "image", "script", "foreignObject", "use", "a", "style",
    "defs", "symbol",
    "pattern", "mask", "clipPath", "filter",
    "linearGradient", "radialGradient", "stop",
]

# PowerPoint完全最適版で禁止したい属性（opacity系を中心に）
PROHIBITED_ATTRS = [
    "opacity", "fill-opacity", "stroke-opacity",
    "filter", "mask", "clip-path",
]

# 禁止したいCSS/スタイル記法（url(...) / rgba等）
PROHIBITED_STYLE_PATTERNS = [
    r"url\(",          # url(...)
    r"rgba\(",         # rgba(...)
    r"hsla\(",         # hsla(...)
]


class SVGValidationError:
    """Container for SVG validation error details."""
    def __init__(self, error_type: str, message: str, severity: str = "error"):
        self.error_type = error_type
        self.message = message
        self.severity = severity  # "error" or "warning"

    def __str__(self):
        return f"[{self.severity.upper()}] {self.error_type}: {self.message}"


def _strip_xml_decl(svg_content: str) -> str:
    s = svg_content.strip()
    if s.startswith("<?xml") and "?>" in s:
        return s[s.index("?>") + 2:].strip()
    return svg_content


def sanitize_svg_remove_external_refs(svg_content: str) -> str:
    """
    Safe sanitizer:
      - removes <image> blocks
      - removes href/xlink:href that point to http(s)
      - neutralizes url(http...) patterns
    IMPORTANT:
      - DO NOT globally strip 'http://' because it breaks xmlns
      - normalize xmlns to the correct SVG namespace
    """
    if not svg_content:
        return svg_content

    s = svg_content

    # remove any <image ... /> or <image ...>...</image>
    s = re.sub(r"<image\b[^>]*(?:/?>|>[\s\S]*?</image>)", "", s, flags=re.IGNORECASE)

    # drop href="http..." and xlink:href="http..."
    s = re.sub(r'\s(?:xlink:href|href)=["\']https?://[^"\']*["\']', "", s, flags=re.IGNORECASE)

    # neutralize url(http...) in styles
    s = re.sub(r'url\((["\']?)https?://.*?\1\)', "none", s, flags=re.IGNORECASE)

    # ★ ここが重要：xmlns を必ず正しいURIに戻す（sanitizeで壊れた場合の復旧）
    s = re.sub(r'xmlns="[^"]*w3\.org/2000/svg"', 'xmlns="http://www.w3.org/2000/svg"', s, flags=re.IGNORECASE)
    s = re.sub(r'xmlns:xlink="[^"]*w3\.org/1999/xlink"', 'xmlns:xlink="http://www.w3.org/1999/xlink"', s, flags=re.IGNORECASE)


    # ensure XML declaration (warning回避)
    if not s.lstrip().startswith("<?xml"):
        s = '<?xml version="1.0" encoding="UTF-8"?>\n' + s

    return s



def _iter_children_elements(root):
    """Return direct child elements of root (skip comments/processing-instructions)."""
    return [c for c in list(root) if isinstance(c.tag, str)]


def _is_solid_background_rect(el: ET.Element) -> bool:
    """Check if element is a full-canvas rect with solid fill and no opacity."""
    if not str(el.tag).endswith("rect"):
        return False

    x = (el.attrib.get("x", "") or "").strip()
    y = (el.attrib.get("y", "") or "").strip()
    w = (el.attrib.get("width", "") or "").strip()
    h = (el.attrib.get("height", "") or "").strip()
    fill = (el.attrib.get("fill", "") or "").strip()

    # width/height は 100% 推奨（PPTで安定）だが、数値でも許容
    full_w = (w == "100%" or w == str(SVG_WIDTH) or w == f"{SVG_WIDTH}px")
    full_h = (h == "100%" or h == str(SVG_HEIGHT) or h == f"{SVG_HEIGHT}px")

    if not full_w or not full_h:
        return False

    # x,y は 0 が望ましい（空なら許容しない）
    if x not in ("0", "0px", ""):
        return False
    if y not in ("0", "0px", ""):
        return False

    # fill must be solid HEX #RRGGBB (no none)
    if not re.fullmatch(r"#([0-9a-fA-F]{6})", fill or ""):
        return False

    # opacity-related attrs prohibited
    for a in PROHIBITED_ATTRS:
        if a in el.attrib:
            return False

    # style attribute should not include prohibited patterns
    st = (el.attrib.get("style") or "")
    if st:
        # forbid opacity even in style
        if re.search(r"(^|;)\s*opacity\s*:", st, flags=re.IGNORECASE):
            return False
        if re.search(r"(^|;)\s*fill-opacity\s*:", st, flags=re.IGNORECASE):
            return False
        if re.search(r"(^|;)\s*stroke-opacity\s*:", st, flags=re.IGNORECASE):
            return False

    return True


def validate_svg_structure(svg_content: str) -> Tuple[bool, List[SVGValidationError]]:
    """
    Validate SVG structure and compliance with PowerPoint-friendly rules.
    """
    errors: List[SVGValidationError] = []

    # Check 1: Non-empty content
    if not svg_content or not svg_content.strip():
        errors.append(SVGValidationError("empty_content", "SVG content is empty or whitespace only"))
        return False, errors

    # Check 2: Contains XML declaration
    if not svg_content.strip().startswith('<?xml'):
        errors.append(SVGValidationError(
            "missing_xml_declaration",
            "SVG missing XML declaration (<?xml version=...)",
            severity="warning"
        ))

    # Check 3: Contains <svg> root element
    if '<svg' not in svg_content.lower():
        errors.append(SVGValidationError("missing_svg_root", "No <svg> root element found"))
        return False, errors

    # Check 3.5: Hard forbid comments
    if "<!--" in svg_content:
        errors.append(SVGValidationError("comments_forbidden", "SVG contains comments <!-- ... --> (forbidden)"))

    # Check 3.6: Hard forbid url(...) and rgba/hsla in entire text
    low = svg_content.lower()
    for pat in PROHIBITED_STYLE_PATTERNS:
        if re.search(pat, low):
            errors.append(SVGValidationError("prohibited_style_syntax", f"Found prohibited style syntax matching: {pat}"))
            break

    # Check 3.7: Hard forbid 8-digit hex (#RRGGBBAA) which can imply alpha
    if re.search(r"#([0-9a-fA-F]{8})", svg_content):
        errors.append(SVGValidationError("alpha_hex_forbidden", "Found 8-digit hex color (#RRGGBBAA) (forbidden)"))

    # Check 4: Basic XML parsing
    try:
        svg_for_parsing = _strip_xml_decl(svg_content)
        root = ET.fromstring(svg_for_parsing)

        if not root.tag.endswith('svg'):
            errors.append(SVGValidationError("invalid_root", f"Root element is {root.tag}, expected <svg>"))
            return False, errors

    except ET.ParseError as e:
        errors.append(SVGValidationError("xml_parse_error", f"XML parsing failed: {str(e)}"))
        return False, errors

    # Check 5: Required attributes on <svg>
    svg_attrs = root.attrib
    if 'width' not in svg_attrs:
        errors.append(SVGValidationError("missing_width", "<svg> missing width attribute"))
    if 'height' not in svg_attrs:
        errors.append(SVGValidationError("missing_height", "<svg> missing height attribute"))
    if 'viewBox' not in svg_attrs:
        errors.append(SVGValidationError("missing_viewbox", "<svg> missing viewBox attribute", severity="warning"))

    # Check 6: Canvas dimensions
    try:
        width_val = int(str(svg_attrs.get('width', '')).replace('px', '').strip())
        height_val = int(str(svg_attrs.get('height', '')).replace('px', '').strip())
        if width_val != SVG_WIDTH or height_val != SVG_HEIGHT:
            errors.append(SVGValidationError(
                "incorrect_dimensions",
                f"Canvas dimensions {width_val}x{height_val} != expected {SVG_WIDTH}x{SVG_HEIGHT}",
                severity="warning"
            ))
    except Exception:
        pass

    # Check 7: Presence of <text> elements (REQUIRED) — namespace-agnostic
    text_elements = [el for el in root.iter() if str(el.tag).endswith("text")]
    if not text_elements:
        errors.append(SVGValidationError(
            "no_text_elements",
            "No <text> elements found - all text must use <text>, not <path>"
        ))

    # Check 8: Prohibited elements
    for prohibited in PROHIBITED_ELEMENTS:
        found = [el for el in root.iter() if str(el.tag).endswith(prohibited)]
        if found:
            errors.append(SVGValidationError(
                "prohibited_element",
                f"Prohibited element <{prohibited}> found in SVG"
            ))

    # Check 9: No base64 data
    if 'base64' in low:
        errors.append(SVGValidationError("base64_found", "Base64 encoded data found - external resources not allowed"))

    # Check 10: No external references (ALLOW SVG/XMLNS namespaces)
    low = svg_content.lower()

    # Allow-list: SVG namespace URIs that MUST be present for PowerPoint compatibility
    allowed_ns = [
        "http://www.w3.org/2000/svg",
        "http://www.w3.org/1999/xlink",
    ]

    # Find all http(s):// occurrences
    urls = re.findall(r'https?://[^\s"\'<>)]+' , low)

    # Remove allowed namespace URIs from the URL list
    urls = [u for u in urls if not any(u.startswith(a) for a in allowed_ns)]

    if urls:
        errors.append(SVGValidationError(
            "external_reference",
            f"External URL references found (not allowed): {urls[:3]}{'...' if len(urls)>3 else ''}"
        ))


    # Check 11: Font family validation (warning)
    if 'font-family' in low and 'meiryo' not in low:
        errors.append(SVGValidationError(
            "incorrect_font",
            f"Font family should include '{FONT_FAMILY}'",
            severity="warning"
        ))

    # Check 12: Forbid opacity anywhere (attribute or style)
    #  - PowerPointで透明化の主要原因
    if re.search(r"\bopacity\s*=", low) or re.search(r"\bfill-opacity\s*=", low) or re.search(r"\bstroke-opacity\s*=", low):
        errors.append(SVGValidationError("opacity_forbidden", "Found opacity attribute (forbidden)"))
    if re.search(r"(^|;)\s*opacity\s*:", low) or re.search(r"(^|;)\s*fill-opacity\s*:", low) or re.search(r"(^|;)\s*stroke-opacity\s*:", low):
        errors.append(SVGValidationError("opacity_forbidden", "Found opacity in style attribute (forbidden)"))

    # Check 13: Enforce solid background rect FIRST
    children = _iter_children_elements(root)
    if not children:
        errors.append(SVGValidationError("no_children", "SVG has no child elements"))
    else:
        first = children[0]
        if not _is_solid_background_rect(first):
            errors.append(SVGValidationError(
                "background_rect_required",
                "First element must be a full-canvas solid <rect ... fill=\"#RRGGBB\"/> without opacity"
            ))

    critical_errors = [e for e in errors if e.severity == "error"]
    return (len(critical_errors) == 0), errors


def validate_all_generated_svgs(
    svg_results: Dict[int, Tuple[Optional[str], bool, List[str]]]
) -> Dict[int, Tuple[bool, List[SVGValidationError]]]:

    validation_results: Dict[int, Tuple[bool, List[SVGValidationError]]] = {}

    print(f"\n{'='*80}")
    print(f"Starting SVG Validation")
    print(f"{'='*80}\n")

    for slide_num, (svg_content, generation_success, gen_errors) in svg_results.items():
        print(f"Validating Slide {slide_num}...")

        if not generation_success or not svg_content:
            print(f"  ⊘ Skipped (generation failed)")
            validation_results[slide_num] = (False, [
                SVGValidationError("generation_failed", "SVG generation did not complete successfully")
            ])
            continue

        # sanitize first (外部参照やコメントを先に除去)
        sanitized = sanitize_svg_remove_external_refs(svg_content)

        is_valid, validation_errors = validate_svg_structure(sanitized)

        if is_valid:
            print(f"  ✓ Valid")
            warns = [e for e in validation_errors if e.severity == "warning"]
            if warns:
                print(f"    ⚠ {len(warns)} warning(s)")
        else:
            err_count = len([e for e in validation_errors if e.severity == "error"])
            warn_count = len([e for e in validation_errors if e.severity == "warning"])
            print(f"  ❌ Invalid ({err_count} error(s), {warn_count} warning(s))")
            for err in validation_errors:
                if err.severity == "error":
                    print(f"      - {err}")

        validation_results[slide_num] = (is_valid, validation_errors)

    total = len(validation_results)
    valid_count = sum(1 for is_valid, _ in validation_results.values() if is_valid)

    print(f"\n{'='*80}")
    print(f"Validation Complete")
    print(f"{'='*80}")
    print(f"✓ Valid: {valid_count}/{total}")
    if valid_count != total:
        print(f"❌ Invalid: {total - valid_count}/{total}")
    print(f"{'='*80}\n")

    return validation_results





In [60]:
# ============================================================
# Cell 07 — Gemini SVG generation + sanitize + validate-aware retry
# ============================================================
# Overview:
#   Generates SVGs for each planned slide using Gemini, extracts SVG from responses,
#   sanitizes unsafe/external references, validates against PowerPoint-friendly rules,
#   and retries generation with validation feedback up to a configured max.
#   Produces per-slide generation results and validation summaries.
#
# Inputs / Outputs:
#   Inputs:
#     - slide_plan_global (List[SlideContent])
#     - gemini_client + GEMINI_MODEL_NAME / GEMINI_MAX_RETRIES
#     - build_svg_prompt / build_validation_prompt / extract_svg_from_response (Cell 05)
#     - sanitize_svg_remove_external_refs / validate_svg_structure / validate_all_generated_svgs (Cell 06)
#   Outputs:
#     - generated_svgs: Dict[int, (svg_content|None, success:bool, errors:list[str])]
#     - svg_validation_results: Dict[int, (is_valid:bool, errors:list[SVGValidationError])]
#     - Console debug logs (response head/tail, progress, retry/validation status)
#
# Notes:
#   - Retries are triggered both on API/extraction failure and on validation failure.
#   - Sanitization + namespace normalization (ensure_svg_xmlns_http) improves PPT import stability.
#   - Final validation is run again to keep a reportable result for downstream steps.
#


import time
from typing import Optional, Tuple, List, Dict


def _errors_to_messages(validation_errors) -> List[str]:
    msgs = []
    for e in validation_errors:
        sev = getattr(e, "severity", "error")
        et = getattr(e, "error_type", "unknown")
        msg = getattr(e, "message", str(e))
        msgs.append(f"{sev}:{et}:{msg}")
    return msgs


def _gemini_generate(prompt: str):
    """
    SDK差異吸収:
      - google-genai: Client.models.generate_content(model=..., contents=...)
      - google-generativeai: GenerativeModel.generate_content(prompt)
    """
    if hasattr(gemini_client, "models") and hasattr(gemini_client.models, "generate_content"):
        return gemini_client.models.generate_content(
            model=GEMINI_MODEL_NAME,   # 例: "models/gemini-2.5-flash"
            contents=prompt,
        )
    if hasattr(gemini_client, "generate_content"):
        return gemini_client.generate_content(prompt)
    raise AttributeError("Unknown Gemini client type: no generate_content method found.")


def _get_response_text(response) -> str:
    """
    response.text が無いSDKもあるので安全に拾う。
    ※ str(response) はメタ情報を含みやすいので、まず text を優先。
    """
    t = getattr(response, "text", None)
    if isinstance(t, str) and t.strip():
        return t
    return str(response)


def generate_svg_with_gemini(
    slide: SlideContent,
    retry_count: int = 0,
    previous_errors: Optional[List[str]] = None,
    previous_svg: str = ""
) -> Tuple[Optional[str], bool, List[str]]:

    if retry_count == 0 or previous_errors is None:
        prompt = build_svg_prompt(slide)
        print(f"\n📝 Generating SVG for Slide {slide.slide_number}: {slide.title}")
    else:
        prompt = build_validation_prompt(slide, previous_svg, previous_errors)
        print(f"\n🔄 Retry {retry_count} for Slide {slide.slide_number}")

    try:
        response = _gemini_generate(prompt)
        response_text = _get_response_text(response)

        # debug: head/tail だけ（Notebookが重くならない）
        print("\n--- DEBUG: response_text head ---")
        print(response_text[:200])
        print("--- DEBUG: response_text tail ---")
        print(response_text[-200:])
        print("----------------------------------\n")

        if not response_text.strip():
            return None, False, ["Empty response from Gemini API"]

        svg_content = extract_svg_from_response(response_text)
        if not svg_content:
            return None, False, [
                "Failed to extract a complete SVG from response",
                f"Response preview: {response_text[:250]}..."
            ]

        print(f"✓ SVG extracted ({len(svg_content)} characters)")
        return svg_content, True, []

    except Exception as e:
        error_msg = f"Gemini API error: {str(e)}"
        print(f"❌ {error_msg}")
        return None, False, [error_msg]


def generate_svg_with_retry(
    slide: SlideContent,
    max_retries: int = GEMINI_MAX_RETRIES
) -> Tuple[Optional[str], bool, List[str], int]:

    previous_errors: Optional[List[str]] = None
    previous_svg: str = ""

    for attempt in range(max_retries):
        svg_content, ok, errors = generate_svg_with_gemini(
            slide,
            retry_count=attempt,
            previous_errors=previous_errors,
            previous_svg=previous_svg
        )

        if not ok or not svg_content:
            previous_errors = errors
            if attempt < max_retries - 1:
                print("⚠ Generation failed, will retry...")
                time.sleep(1.5)
            continue

        # sanitize → validate
        sanitized = sanitize_svg_remove_external_refs(svg_content)
        is_valid, validation_errors = validate_svg_structure(sanitized)

        if slide.slide_number == 1:
            print("\n--- DEBUG: validation errors (slide 1) ---")
            for e in validation_errors:
                print(str(e))
            print("----------------------------------------\n")

        if is_valid:
            return sanitized, True, [], attempt + 1

        previous_svg = sanitized
        previous_errors = _errors_to_messages(validation_errors)

        if attempt < max_retries - 1:
            print("⚠ Validation failed, will retry with errors...")
            time.sleep(1.5)

    return None, False, previous_errors or ["All retry attempts failed"], max_retries


def generate_all_slides_svg(
    slide_plan: List[SlideContent]
) -> Dict[int, Tuple[Optional[str], bool, List[str]]]:

    results: Dict[int, Tuple[Optional[str], bool, List[str]]] = {}
    total_slides = len(slide_plan)

    print(f"\n{'='*80}")
    print(f"Starting SVG generation for {total_slides} slides")
    print(f"{'='*80}")

    for idx, slide in enumerate(slide_plan, 1):
        print(f"\n[{idx}/{total_slides}] Processing Slide {slide.slide_number}...")

        svg_content, success, errors, attempts = generate_svg_with_retry(slide)

        if success:
            print(f"✓ Slide {slide.slide_number} generated + validated (attempts: {attempts})")
            results[slide.slide_number] = (svg_content, True, [])
        else:
            print(f"❌ Slide {slide.slide_number} failed after {attempts} attempts")
            for error in errors:
                print(f"   - {error}")
            results[slide.slide_number] = (None, False, errors)

        if idx < total_slides:
            time.sleep(0.5)

    successful = sum(1 for _, (_, s, _) in results.items() if s)
    print(f"\n{'='*80}")
    print("SVG Generation Complete")
    print(f"{'='*80}")
    print(f"✓ Successful: {successful}/{total_slides}")
    print(f"❌ Failed: {total_slides - successful}/{total_slides}" if successful != total_slides else "")
    print(f"{'='*80}\n")

    return results

def ensure_svg_xmlns_http(svg_content: str) -> str:
    """
    PowerPoint対策：
    - xmlns / xmlns:xlink が http:// なしや https:// になっていたら必ず正規の http:// に戻す
    - ついでに <svg ...> に xmlns が無い場合も補う
    """
    if not svg_content or "<svg" not in svg_content.lower():
        return svg_content

    s = svg_content

    # 1) xmlns="www.w3.org/2000/svg" などを強制的に http:// にする
    s = re.sub(
        r'xmlns="(?:https?://)?www\.w3\.org/2000/svg"',
        'xmlns="http://www.w3.org/2000/svg"',
        s,
        flags=re.IGNORECASE
    )

    # 2) xmlns:xlink も同様に
    s = re.sub(
        r'xmlns:xlink="(?:https?://)?www\.w3\.org/1999/xlink"',
        'xmlns:xlink="http://www.w3.org/1999/xlink"',
        s,
        flags=re.IGNORECASE
    )

    # 3) そもそも xmlns が無い <svg> が来た場合の保険（あまり無いけど）
    #    <svg ...> の最初のタグに xmlns を足す（既に xmlns があれば何もしない）
    if re.search(r"<svg\b[^>]*\bxmlns=", s, flags=re.IGNORECASE) is None:
        s = re.sub(
            r"<svg\b",
            '<svg xmlns="http://www.w3.org/2000/svg"',
            s,
            count=1,
            flags=re.IGNORECASE
        )

    # 4) XML宣言が無ければ付ける（warning回避）
    if not s.lstrip().startswith("<?xml"):
        s = '<?xml version="1.0" encoding="UTF-8"?>\n' + s

    return s
# --- Main execution ---
# --- Main execution (recommended) ---
print("Initializing SVG generation workflow...\n")
generated_svgs = generate_all_slides_svg(slide_plan_global)

sanitized_svgs = {}
for slide_num, (svg_content, gen_ok, gen_errors) in generated_svgs.items():
    if gen_ok and svg_content:
        svg_content = sanitize_svg_remove_external_refs(svg_content)
        svg_content = ensure_svg_xmlns_http(svg_content)
    sanitized_svgs[slide_num] = (svg_content, gen_ok, gen_errors)

print("Running validation...\n")
svg_validation_results = validate_all_generated_svgs(sanitized_svgs)

print("\nStarting SVG validation workflow...\n")
# ここは最終チェック用（retry内でもvalidateしてるが、レポート/後続のために保持）
svg_validation_results = validate_all_generated_svgs(generated_svgs)

print("\n✓ Cell 09 complete")
print("✓ generated_svgs and svg_validation_results are ready")


Initializing SVG generation workflow...


Starting SVG generation for 7 slides

[1/7] Processing Slide 1...

📝 Generating SVG for Slide 1: B Capitalについて

--- DEBUG: response_text head ---
<?xml version="1.0" encoding="UTF-8"?>
<svg width="1920" height="1080" viewBox="0 0 1920 1080" xmlns="http://www.w3.org/2000/svg">
<rect x="0" y="0" width="1920" height="1080" fill="#0B3D91"/>
<text x
--- DEBUG: response_text tail ---
nt-weight="400" text-anchor="middle" fill="#FFFFFF">2026/2/13</text>
<text x="1800" y="960" font-family="Meiryo" font-size="22" text-anchor="end" fill="#FFFFFF">B Capital Japan &amp; BCG</text>
</svg>
----------------------------------

✓ SVG extracted (586 characters)

--- DEBUG: validation errors (slide 1) ---
----------------------------------------

✓ Slide 1 generated + validated (attempts: 1)

[2/7] Processing Slide 2...

📝 Generating SVG for Slide 2: B Capitalについて

--- DEBUG: response_text head ---
<?xml version="1.0" encoding="UTF-8"?>
<svg width="1920" height="1

In [61]:
# ============================================================
# Cell 08 — Save validated SVG files to disk (fix xmlns to http://...)
# ============================================================
# Overview:
#   Persists generated SVGs to disk after validation. Before saving, normalizes SVG
#   namespaces (xmlns / xmlns:xlink) to the canonical http:// URIs for PowerPoint
#   stability. Also verifies saved files and writes a JSON manifest for downstream use.
#
# Inputs / Outputs:
#   Inputs:
#     - sanitized_svgs (Dict[int, (svg_content|None, gen_ok:bool, gen_errors:list[str])])
#     - svg_validation_results (Dict[int, (is_valid:bool, errors:list[SVGValidationError])])
#     - SVG_DIR (output directory), _ts_ms(), ensure_svg_xmlns_http()
#   Outputs:
#     - saved_svg_results / saved_files: Dict[int, (saved_ok:bool, path|None, message:str)]
#     - Written SVG files in SVG_DIR (timestamped names)
#     - Verification status + list of issues (if any)
#     - svg_manifest.json in SVG_DIR describing saved files and sizes
#
# Notes:
#   - Cleans existing slide_*.svg files in the output directory before saving new ones.
#   - By default saves only validated SVGs; optional save_invalid can save failed SVGs for debugging.
#   - Verification includes a namespace normalization check to avoid PPT transparency/render issues.
#


from typing import Dict, List, Tuple, Optional
from pathlib import Path
import json
import time
import re


def save_svg_to_file(svg_content: str, slide_number: int, output_dir: Path) -> Tuple[bool, Optional[Path], str]:
    """Save SVG content to disk with timestamped naming to avoid overwrites."""
    try:
        ts = _ts_ms()  # ← ここで日時
        filename = f"slide_{slide_number:03d}_{ts}.svg"
        file_path = Path(output_dir) / filename
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(svg_content)
        return True, file_path, ""
    except Exception as e:
        return False, None, f"Failed to save slide {slide_number}: {e}"



def save_all_validated_svgs(
    svg_results: Dict[int, Tuple[Optional[str], bool, List[str]]],
    validation_results: Dict[int, Tuple[bool, List["SVGValidationError"]]],
    output_dir: Path,
    save_invalid: bool = False
) -> Dict[int, Tuple[bool, Optional[Path], str]]:
    """
    Save SVGs that passed validation.
    If save_invalid=True, invalid ones are saved as slide_XXX_INVALID.svg for debugging.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*80}")
    print(f"Saving SVG files to: {output_dir}")
    print(f"{'='*80}\n")

    # Clean existing
    for existing in output_dir.glob("slide_*.svg"):
        existing.unlink()
    print(f"✓ Cleaned existing SVG files from {output_dir}\n")

    save_results: Dict[int, Tuple[bool, Optional[Path], str]] = {}

    total = len(svg_results)
    saved = 0
    skipped = 0
    errors = 0

    for slide_num in sorted(svg_results.keys()):
        svg_content, gen_ok, gen_errors = svg_results[slide_num]
        is_valid, val_errors = validation_results.get(slide_num, (False, []))

        print(f"Slide {slide_num:02d}: ", end="")

        if not gen_ok or not svg_content:
            print("❌ Skipped (generation failed)")
            save_results[slide_num] = (False, None, "Generation failed")
            skipped += 1
            continue

        if not is_valid:
            if save_invalid:
                filename = f"slide_{slide_num:03d}_INVALID.svg"
                file_path = output_dir / filename
                try:
                    # ★ INVALID保存でも xmlns を直しておくと後で比較しやすい
                    fixed = ensure_svg_xmlns_http(svg_content)
                    with open(file_path, "w", encoding="utf-8") as f:
                        f.write(fixed)
                    print(f"⚠ Saved INVALID (debug): {filename}")
                    save_results[slide_num] = (False, file_path, "Saved invalid for debugging")
                except Exception as e:
                    print(f"❌ Failed to save invalid: {e}")
                    save_results[slide_num] = (False, None, str(e))
                    errors += 1
                skipped += 1
            else:
                err_cnt = len([e for e in val_errors if getattr(e, "severity", "error") == "error"])
                print(f"❌ Skipped (validation failed: {err_cnt} error(s))")
                save_results[slide_num] = (False, None, "Validation failed")
                skipped += 1
            continue

        # ★★ ここが差し替えの本体：保存直前に xmlns を必ず http:// に戻す ★★
        svg_content_fixed = ensure_svg_xmlns_http(svg_content)

        ok, file_path, msg = save_svg_to_file(svg_content_fixed, slide_num, output_dir)
        if ok and file_path:
            kb = file_path.stat().st_size / 1024
            print(f"✓ Saved ({kb:.1f} KB): {file_path.name}")
            save_results[slide_num] = (True, file_path, "")
            saved += 1
        else:
            print(f"❌ Error: {msg}")
            save_results[slide_num] = (False, None, msg)
            errors += 1

    print(f"\n{'='*80}")
    print("Save Operation Complete")
    print(f"{'='*80}")
    print(f"✓ Saved successfully: {saved}/{total}")
    if skipped:
        print(f"⚠ Skipped (not saved): {skipped}/{total}")
    if errors:
        print(f"❌ Save errors: {errors}/{total}")
    print(f"{'='*80}\n")

    return save_results


def verify_saved_svgs(save_results: Dict[int, Tuple[bool, Optional[Path], str]]) -> Tuple[bool, List[str]]:
    """Verify saved SVG files exist and contain <svg>."""
    issues: List[str] = []

    print("Verifying saved SVG files...\n")

    for slide_num, (ok, path, msg) in save_results.items():
        if not ok:
            issues.append(f"Slide {slide_num}: Not saved - {msg}")
            continue
        if not path or not Path(path).exists():
            issues.append(f"Slide {slide_num}: File not found - {path}")
            continue

        try:
            content = Path(path).read_text(encoding="utf-8")
            if not content.strip():
                issues.append(f"Slide {slide_num}: File empty")
            elif "<svg" not in content.lower():
                issues.append(f"Slide {slide_num}: Missing <svg> tag")

            # 追加チェック：xmlns が http:// になっているか（透明化の主要因）
            if 'xmlns="http://www.w3.org/2000/svg"' not in content:
                issues.append(f"Slide {slide_num}: xmlns is not normalized to http://www.w3.org/2000/svg")

        except Exception as e:
            issues.append(f"Slide {slide_num}: Read error - {e}")

    all_ok = (len(issues) == 0)
    if all_ok:
        print("✓ All saved SVG files verified successfully")
    else:
        print(f"⚠ Verification found {len(issues)} issue(s):")
        for it in issues:
            print(f"   - {it}")

    return all_ok, issues


def create_save_manifest(save_results: Dict[int, Tuple[bool, Optional[Path], str]], output_dir: Path) -> Path:
    """Create svg_manifest.json in output_dir."""
    output_dir = Path(output_dir)
    manifest_path = output_dir / "svg_manifest.json"

    manifest = {
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
        "total_slides": len(save_results),
        "files": []
    }

    for slide_num in sorted(save_results.keys()):
        ok, path, msg = save_results[slide_num]
        entry = {"slide_number": slide_num, "saved": bool(ok)}
        if ok and path:
            p = Path(path)
            entry.update({
                "filename": p.name,
                "path": str(p),
                "size_bytes": p.stat().st_size
            })
        else:
            entry["error"] = msg
        manifest["files"].append(entry)

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    return manifest_path


# --- Main execution ---
print("Starting SVG file save operation...\n")

saved_svg_results = save_all_validated_svgs(
    svg_results=sanitized_svgs,                 # sanitize済み
    validation_results=svg_validation_results,  # validate結果
    output_dir=SVG_DIR,
    save_invalid=False
)

verification_passed, verification_issues = verify_saved_svgs(saved_svg_results)

manifest_path = create_save_manifest(saved_svg_results, SVG_DIR)
print(f"\n✓ Manifest created: {manifest_path}")

saved_files = saved_svg_results

print("\n✓ SVG file save operation complete")
print("✓ Files ready for PowerPoint embedding")


Starting SVG file save operation...


Saving SVG files to: output/run_20260213_091919/svgs

✓ Cleaned existing SVG files from output/run_20260213_091919/svgs

Slide 01: ✓ Saved (0.6 KB): slide_001_20260213_094104_918.svg
Slide 02: ✓ Saved (1.4 KB): slide_002_20260213_094104_919.svg
Slide 03: ✓ Saved (2.1 KB): slide_003_20260213_094104_920.svg
Slide 04: ✓ Saved (1.7 KB): slide_004_20260213_094104_920.svg
Slide 05: ✓ Saved (1.7 KB): slide_005_20260213_094104_921.svg
Slide 06: ✓ Saved (2.7 KB): slide_006_20260213_094104_921.svg
Slide 07: ✓ Saved (1.6 KB): slide_007_20260213_094104_921.svg

Save Operation Complete
✓ Saved successfully: 7/7

Verifying saved SVG files...

✓ All saved SVG files verified successfully

✓ Manifest created: output/run_20260213_091919/svgs/svg_manifest.json

✓ SVG file save operation complete
✓ Files ready for PowerPoint embedding


In [62]:
# ============================================================
# Cell 09 — PowerPoint assembly with SVG embedding (cairosvg + namespace fix)
# ============================================================
# Overview:
#   Builds a PPTX by converting each saved SVG to a PNG (via cairosvg) and embedding it
#   as a full-slide image using python-pptx. Includes a conversion-time SVG namespace
#   normalization step for cairosvg stability, then validates the output PPTX and writes
#   a metadata JSON report.
#
# Inputs / Outputs:
#   Inputs:
#     - saved_files: Dict[int, (ok:bool, svg_path|None, message:str)] from Cell 08
#     - SVG_WIDTH, SVG_HEIGHT (target render size in px)
#     - PPTX_OUTPUT, OUTPUT_DIR (output paths)
#     - cairosvg (installed), python-pptx
#   Outputs:
#     - PPTX file written to PPTX_OUTPUT
#     - powerpoint_output dict: {success, path, metadata, errors}
#     - pptx_metadata.json written under OUTPUT_DIR (size, slide count, validation/build errors)
#     - Console progress logs for slide add / skips / final file stats
#
# Notes:
#   - SVGs are embedded as raster images (PNG); resulting PPTX is not shape-editable.
#   - `_normalize_svg_for_cairosvg` strips xmlns/xlink for conversion robustness only.
#   - Slides use a blank layout and place the PNG to cover the entire slide area.
#   - `validate_pptx_output` ensures the PPTX is readable and contains slides.
#


from pptx import Presentation
from pptx.util import Inches
import io
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import json
import time
import re

# 前セルで定義済み想定：
#   SVG_WIDTH, SVG_HEIGHT, SVG_DIR, PPTX_OUTPUT, OUTPUT_DIR
#   saved_files: Dict[int, Tuple[bool, Optional[Path], str]]

def create_blank_presentation() -> Presentation:
    prs = Presentation()
    prs.slide_width = Inches(16)   # 1920px相当
    prs.slide_height = Inches(9)   # 1080px相当
    return prs


def _normalize_svg_for_cairosvg(svg: str) -> str:
    """
    cairosvg が名前空間付きタグを誤判定する環境があるため、
    変換専用に xmlns / xlink 等を落としてから渡す。
    """
    if not svg:
        return svg

    s = svg.strip()

    # XML宣言は残してもOKだが、念のため先頭に寄せる
    # （中にBOM等あると壊れることがある）
    s = s.lstrip("\ufeff")

    # ルート <svg ...> の xmlns / xmlns:xlink を除去（変換用）
    s = re.sub(r'\sxmlns(:\w+)?="[^"]+"', "", s, flags=re.IGNORECASE)

    # 念のため xlink:href の名前空間接頭辞が残っていても良いように
    # 属性名だけ普通の href に寄せる（ある場合のみ）
    s = re.sub(r'\bxlink:href=', "href=", s, flags=re.IGNORECASE)

    return s


def svg_to_png_bytes(svg_content: str, width: int, height: int) -> bytes:
    """
    SVG -> PNG を cairosvg で行う。
    """
    import cairosvg  # ここで import（環境にある前提）

    normalized = _normalize_svg_for_cairosvg(svg_content)

    try:
        return cairosvg.svg2png(
            bytestring=normalized.encode("utf-8"),
            output_width=width,
            output_height=height,
        )
    except Exception as e:
        # デバッグしやすいように、先頭だけ添える
        head = normalized[:200].replace("\n", "\\n")
        raise RuntimeError(f"cairosvg failed: {e}\nSVG head: {head}")


def add_svg_slide_to_presentation(
    prs: Presentation,
    svg_path: Path,
    slide_number: int,
    width_px: int,
    height_px: int,
) -> Tuple[bool, str]:
    try:
        svg_content = Path(svg_path).read_text(encoding="utf-8")
        png_bytes = svg_to_png_bytes(svg_content, width_px, height_px)

        blank_layout = prs.slide_layouts[6]
        slide = prs.slides.add_slide(blank_layout)

        slide.shapes.add_picture(
            io.BytesIO(png_bytes),
            left=0,
            top=0,
            width=prs.slide_width,
            height=prs.slide_height,
        )
        return True, ""
    except Exception as e:
        return False, f"Failed to add slide {slide_number}: {e}"


def build_powerpoint_from_svgs(
    save_results: Dict[int, Tuple[bool, Optional[Path], str]],
    output_path: Path,
    width_px: int = 1920,
    height_px: int = 1080,
) -> Tuple[bool, Optional[Path], List[str]]:

    errors: List[str] = []

    print(f"\n{'='*80}")
    print("Building PowerPoint presentation")
    print(f"{'='*80}\n")

    prs = create_blank_presentation()
    print(f"✓ Created blank presentation ({prs.slide_width.inches}x{prs.slide_height.inches} inches)")

    sorted_slides = sorted(save_results.items(), key=lambda x: x[0])

    added_count = 0
    skipped_count = 0

    for slide_num, (ok, svg_path, err) in sorted_slides:
        print(f"Slide {slide_num:02d}: ", end="")

        if not ok or svg_path is None:
            print("❌ Skipped (no valid SVG file)")
            errors.append(f"Slide {slide_num}: {err}")
            skipped_count += 1
            continue

        slide_ok, slide_err = add_svg_slide_to_presentation(
            prs, svg_path, slide_num, width_px, height_px
        )

        if slide_ok:
            print("✓ Added")
            added_count += 1
        else:
            print(f"❌ Failed: {slide_err}")
            errors.append(slide_err)
            skipped_count += 1

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    prs.save(str(output_path))

    size_mb = output_path.stat().st_size / (1024 * 1024)

    print(f"\n{'='*80}")
    print("PowerPoint Assembly Complete")
    print(f"{'='*80}")
    print(f"✓ Slides added: {added_count}")
    if skipped_count:
        print(f"⚠ Slides skipped: {skipped_count}")
    print(f"✓ File saved: {output_path}")
    print(f"✓ File size: {size_mb:.2f} MB")
    print(f"{'='*80}\n")

    success = added_count > 0
    return success, output_path, errors


def validate_pptx_output(pptx_path: Path) -> Tuple[bool, Dict[str, object]]:
    info = {"file_exists": False, "file_size_bytes": 0, "readable": False, "slide_count": 0, "errors": []}

    pptx_path = Path(pptx_path)
    if not pptx_path.exists():
        info["errors"].append(f"File not found: {pptx_path}")
        return False, info

    info["file_exists"] = True
    info["file_size_bytes"] = pptx_path.stat().st_size

    try:
        prs = Presentation(str(pptx_path))
        info["readable"] = True
        info["slide_count"] = len(prs.slides)
        if info["slide_count"] == 0:
            info["errors"].append("Presentation contains no slides")
    except Exception as e:
        info["errors"].append(f"Failed to read PPTX: {e}")

    return (len(info["errors"]) == 0), info


# --- Main execution ---
print("Starting PowerPoint assembly workflow...\n")

pptx_success, pptx_path, pptx_errors = build_powerpoint_from_svgs(
    save_results=saved_files,
    output_path=PPTX_OUTPUT,
    width_px=SVG_WIDTH,
    height_px=SVG_HEIGHT,
)

pptx_metadata = {}
if pptx_path:
    is_valid, validation_info = validate_pptx_output(pptx_path)
    pptx_metadata = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "output_file": str(pptx_path),
        "file_size_bytes": validation_info.get("file_size_bytes", 0),
        "file_size_mb": validation_info.get("file_size_bytes", 0) / (1024 * 1024),
        "slide_count": validation_info.get("slide_count", 0),
        "valid": is_valid,
        "validation_errors": validation_info.get("errors", []),
        "build_errors": pptx_errors,
    }

    metadata_path = Path(OUTPUT_DIR) / "pptx_metadata.json"
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(pptx_metadata, f, ensure_ascii=False, indent=2)
    print(f"✓ Metadata saved: {metadata_path}")

powerpoint_output = {
    "success": bool(pptx_success),
    "path": pptx_path,
    "metadata": pptx_metadata,
    "errors": pptx_errors,
}

print("\n✓ Cell 11 execution complete")


Starting PowerPoint assembly workflow...


Building PowerPoint presentation

✓ Created blank presentation (16.0x9.0 inches)
Slide 01: ✓ Added
Slide 02: ✓ Added
Slide 03: ✓ Added
Slide 04: ✓ Added
Slide 05: ✓ Added
Slide 06: ✓ Added
Slide 07: ✓ Added

PowerPoint Assembly Complete
✓ Slides added: 7
✓ File saved: output/run_20260213_091919/meeting_presentation_20260213_091919.pptx
✓ File size: 0.22 MB

✓ Metadata saved: output/run_20260213_091919/pptx_metadata.json

✓ Cell 11 execution complete


In [63]:
# ============================================================
# Cell 10 — Output validation and delivery
# ============================================================
# Overview:
#
# Inputs / Outputs:
#
# Notes:
#

from typing import Dict, List, Tuple, Optional
import shutil
from pathlib import Path


def generate_final_report(
    meeting_agenda: Dict,
    svg_generation_results: Dict,
    validation_results: Dict,
    save_results: Dict,
    pptx_output: Dict
) -> Dict:
    """
    Generate comprehensive final report for entire workflow.
    
    Args:
        meeting_agenda: Original meeting agenda data
        svg_generation_results: Results from Cell 08
        validation_results: Results from Cell 09
        save_results: Results from Cell 10
        pptx_output: Results from Cell 11
        
    Returns:
        Dictionary containing complete workflow report
    """
    
    total_slides = len(svg_generation_results)
    
    # SVG generation stats
    gen_successful = sum(1 for _, (_, success, _) in svg_generation_results.items() if success)
    gen_failed = total_slides - gen_successful
    
    # Validation stats
    val_passed = sum(1 for _, (is_valid, _) in validation_results.items() if is_valid)
    val_failed = total_slides - val_passed
    
    # Save stats
    save_successful = sum(1 for _, (success, _, _) in save_results.items() if success)
    save_failed = total_slides - save_successful
    
    # Build final report
    report = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "workflow": "meeting_presentation_generator_svg",
        "version": "1.0",
        "meeting_info": {
            "title": meeting_agenda.get('title', 'N/A'),
            "date": meeting_agenda.get('date', 'N/A'),
            "topics_count": len(meeting_agenda.get('topics', [])),
            "total_slides": total_slides
        },
        "svg_generation": {
            "successful": gen_successful,
            "failed": gen_failed,
            "success_rate": f"{(gen_successful/total_slides*100):.1f}%" if total_slides > 0 else "0%"
        },
        "svg_validation": {
            "passed": val_passed,
            "failed": val_failed,
            "pass_rate": f"{(val_passed/total_slides*100):.1f}%" if total_slides > 0 else "0%"
        },
        "file_save": {
            "successful": save_successful,
            "failed": save_failed,
            "success_rate": f"{(save_successful/total_slides*100):.1f}%" if total_slides > 0 else "0%"
        },
        "powerpoint_output": {
            "generated": pptx_output.get('success', False),
            "file_path": str(pptx_output.get('path', 'N/A')),
            "slide_count": pptx_output.get('metadata', {}).get('slide_count', 0),
            "file_size_mb": pptx_output.get('metadata', {}).get('file_size_mb', 0),
            "errors": pptx_output.get('errors', [])
        },
        "output_files": {
            "svg_directory": str(SVG_DIR),
            "pptx_file": str(PPTX_OUTPUT) if pptx_output.get('success') else None,
            "validation_report": str(VALIDATION_REPORT),
            "svg_manifest": str(SVG_DIR / "svg_manifest.json")
        },
        "overall_success": pptx_output.get('success', False) and val_passed >= (total_slides * 0.8)
    }
    
    return report


def print_final_summary(report: Dict) -> None:
    """
    Print human-readable final summary to console.
    
    Args:
        report: Final workflow report dictionary
    """
    
    print(f"\n{'='*80}")
    print(f"FINAL WORKFLOW SUMMARY")
    print(f"{'='*80}\n")
    
    # Meeting info
    print(f"📋 Meeting: {report['meeting_info']['title']}")
    print(f"📅 Date: {report['meeting_info']['date']}")
    print(f"📊 Total Slides: {report['meeting_info']['total_slides']}\n")
    
    # SVG Generation
    print(f"🎨 SVG Generation:")
    print(f"   ✓ Successful: {report['svg_generation']['successful']}")
    if report['svg_generation']['failed'] > 0:
        print(f"   ❌ Failed: {report['svg_generation']['failed']}")
    print(f"   Success Rate: {report['svg_generation']['success_rate']}\n")
    
    # Validation
    print(f"✅ SVG Validation:")
    print(f"   ✓ Passed: {report['svg_validation']['passed']}")
    if report['svg_validation']['failed'] > 0:
        print(f"   ❌ Failed: {report['svg_validation']['failed']}")
    print(f"   Pass Rate: {report['svg_validation']['pass_rate']}\n")
    
    # File Save
    print(f"💾 File Save:")
    print(f"   ✓ Saved: {report['file_save']['successful']}")
    if report['file_save']['failed'] > 0:
        print(f"   ❌ Failed: {report['file_save']['failed']}")
    print(f"   Success Rate: {report['file_save']['success_rate']}\n")
    
    # PowerPoint Output
    print(f"📊 PowerPoint Output:")
    if report['powerpoint_output']['generated']:
        print(f"   ✓ Generated: {report['powerpoint_output']['file_path']}")
        print(f"   ✓ Slides: {report['powerpoint_output']['slide_count']}")
        print(f"   ✓ Size: {report['powerpoint_output']['file_size_mb']:.2f} MB")
    else:
        print(f"   ❌ Generation failed")
        if report['powerpoint_output']['errors']:
            print(f"   Errors:")
            for error in report['powerpoint_output']['errors'][:3]:
                print(f"      - {error}")
    
    print(f"\n📁 Output Files:")
    print(f"   - SVG Directory: {report['output_files']['svg_directory']}")
    if report['output_files']['pptx_file']:
        print(f"   - PowerPoint: {report['output_files']['pptx_file']}")
    print(f"   - Validation Report: {report['output_files']['validation_report']}")
    print(f"   - SVG Manifest: {report['output_files']['svg_manifest']}")
    
    print(f"\n{'='*80}")
    if report['overall_success']:
        print(f"✅ WORKFLOW COMPLETED SUCCESSFULLY")
    else:
        print(f"⚠️  WORKFLOW COMPLETED WITH WARNINGS/ERRORS")
    print(f"{'='*80}\n")


def create_delivery_package(report: Dict, package_dir: Path) -> Tuple[bool, Optional[Path], List[str]]:
    """
    Create a delivery package with all output files.
    
    Args:
        report: Final workflow report
        package_dir: Directory for delivery package
        
    Returns:
        Tuple of (success, package_path, errors)
    """
    
    errors = []
    
    try:
        # Create package directory
        package_dir.mkdir(parents=True, exist_ok=True)
        
        # Copy PowerPoint file
        pptx_source = report['output_files'].get('pptx_file')
        if pptx_source and Path(pptx_source).exists():
            pptx_dest = package_dir / Path(pptx_source).name
            shutil.copy2(pptx_source, pptx_dest)
        else:
            errors.append("PowerPoint file not found")
        
        # Copy SVG directory
        svg_source = Path(report['output_files']['svg_directory'])
        if svg_source.exists():
            svg_dest = package_dir / "svg_slides"
            if svg_dest.exists():
                shutil.rmtree(svg_dest)
            shutil.copytree(svg_source, svg_dest)
        else:
            errors.append("SVG directory not found")
        
        # Copy reports
        val_report_source = Path(report['output_files']['validation_report'])
        if val_report_source.exists():
            shutil.copy2(val_report_source, package_dir / val_report_source.name)
        
        # Save final report
        final_report_path = package_dir / "final_report.json"
        with open(final_report_path, 'w', encoding='utf-8') as f:
            json.dump(report, f, ensure_ascii=False, indent=2)
        
        # Create README
        readme_path = package_dir / "README.txt"
        with open(readme_path, 'w', encoding='utf-8') as f:
            f.write(f"Meeting Presentation Delivery Package\n")
            f.write(f"=====================================\n\n")
            f.write(f"Generated: {report['timestamp']}\n")
            f.write(f"Meeting: {report['meeting_info']['title']}\n")
            f.write(f"Date: {report['meeting_info']['date']}\n\n")
            f.write(f"Contents:\n")
            f.write(f"- {Path(pptx_source).name}: Main presentation file\n")
            f.write(f"- svg_slides/: Individual SVG slide files\n")
            f.write(f"- validation_report.json: SVG validation details\n")
            f.write(f"- final_report.json: Complete workflow report\n\n")
            f.write(f"Notes:\n")
            f.write(f"- Open the .pptx file in PowerPoint/LibreOffice\n")
            f.write(f"- SVG slides are embedded as PNG images\n")
            f.write(f"- Original SVG files available for editing\n")
        
        return True, package_dir, errors
        
    except Exception as e:
        errors.append(f"Package creation failed: {str(e)}")
        return False, None, errors


def validate_delivery_package(package_dir: Path) -> Tuple[bool, List[str]]:
    """
    Validate delivery package completeness.
    
    Args:
        package_dir: Path to delivery package
        
    Returns:
        Tuple of (is_valid, list_of_issues)
    """
    
    issues = []
    
    # Check required files
    required_files = [
        "meeting_presentation.pptx",
        "final_report.json",
        "README.txt"
    ]
    
    for filename in required_files:
        file_path = package_dir / filename
        if not file_path.exists():
            issues.append(f"Missing required file: {filename}")
    
    # Check SVG directory
    svg_dir = package_dir / "svg_slides"
    if not svg_dir.exists():
        issues.append("Missing svg_slides directory")
    elif not list(svg_dir.glob("slide_*.svg")):
        issues.append("svg_slides directory is empty")
    
    is_valid = len(issues) == 0
    return is_valid, issues


# --- Main execution ---
print("\nStarting final validation and delivery workflow...\n")

# Generate final report
final_report = generate_final_report(
    meeting_agenda=meeting_agenda,
    svg_generation_results=generated_svgs,
    validation_results=svg_validation_results,
    save_results=saved_files,
    pptx_output=powerpoint_output
)

# Save final report
final_report_path = OUTPUT_DIR / "final_report.json"
with open(final_report_path, 'w', encoding='utf-8') as f:
    json.dump(final_report, f, ensure_ascii=False, indent=2)

print(f"✓ Final report saved: {final_report_path}")

# Print summary to console
print_final_summary(final_report)

# Create delivery package
package_dir = OUTPUT_DIR / "delivery_package"
print(f"Creating delivery package: {package_dir}\n")

package_success, package_path, package_errors = create_delivery_package(
    final_report,
    package_dir
)

if package_success:
    print(f"✓ Delivery package created: {package_path}\n")
    
    # Validate package
    is_valid, validation_issues = validate_delivery_package(package_path)
    
    if is_valid:
        print(f"✓ Delivery package validated successfully")

else:
    print(f"❌ Delivery package creation failed")
    for error in package_errors:
        print(f"   - {error}")

# Final status
print(f"\n{'='*80}")
if final_report['overall_success'] and package_success:
    print(f"✅ ALL OPERATIONS COMPLETED SUCCESSFULLY")
    print(f"\n📦 Deliverables:")
    print(f"   - PowerPoint: {PPTX_OUTPUT}")
    print(f"   - Delivery Package: {package_dir}")
    print(f"   - Final Report: {final_report_path}")
else:
    print(f"⚠️  WORKFLOW COMPLETED WITH ISSUES")
    print(f"\nPlease review:")
    print(f"   - Final Report: {final_report_path}")
    print(f"   - Validation Report: {VALIDATION_REPORT}")

print(f"{'='*80}\n")

print("✓ Cell 12 execution complete")
print("✓ Notebook workflow finished")



Starting final validation and delivery workflow...

✓ Final report saved: output/run_20260213_091919/final_report.json

FINAL WORKFLOW SUMMARY

📋 Meeting: B Capitalについて
📅 Date: 2026/2/13
📊 Total Slides: 7

🎨 SVG Generation:
   ✓ Successful: 7
   Success Rate: 100.0%

✅ SVG Validation:
   ✓ Passed: 7
   Pass Rate: 100.0%

💾 File Save:
   ✓ Saved: 7
   Success Rate: 100.0%

📊 PowerPoint Output:
   ✓ Generated: output/run_20260213_091919/meeting_presentation_20260213_091919.pptx
   ✓ Slides: 7
   ✓ Size: 0.22 MB

📁 Output Files:
   - SVG Directory: output/run_20260213_091919/svgs
   - PowerPoint: output/run_20260213_091919/meeting_presentation_20260213_091919.pptx
   - Validation Report: output/validation_report_20260213_091824_468.json
   - SVG Manifest: output/run_20260213_091919/svgs/svg_manifest.json

✅ WORKFLOW COMPLETED SUCCESSFULLY

Creating delivery package: output/run_20260213_091919/delivery_package

✓ Delivery package created: output/run_20260213_091919/delivery_package


✅ AL

In [64]:
# ============================================================
# Cell 11 — optional, generate pptx with SVG files
# ============================================================
# Overview:
#   Optionally builds a PowerPoint deck directly via AppleScript by inserting the
#   saved SVG files into new slides. Converts saved SVG file paths from POSIX to HFS
#   format, passes them to AppleScript as a list, then loops to add one slide per SVG.
#
# Inputs / Outputs:
#   Inputs:
#     - saved_files: Dict[int, (ok:bool, path|None, msg:str)] from Cell 08
#     - macOS osascript + Microsoft PowerPoint (installed and scriptable)
#   Outputs:
#     - picked: List[str] of resolved POSIX SVG paths (valid only)
#     - picked_hfs: List[str] of HFS-style paths (Macintosh HD:Users:... format)
#     - A newly created PowerPoint presentation populated with one slide per SVG
#     - Console logs showing picked paths
#
# Notes:
#   - This path inserts SVGs using PowerPoint’s AppleScript interface; behavior can vary by
#     PowerPoint version and SVG features (some SVGs may render differently than PNG embedding).
#   - Uses `POSIX file ... as text` for robust HFS conversion (avoids hardcoding disk names).
#   - If performance is an issue, reduce delays once stability is confirmed.
#   - Slide/image sizing is currently fixed (960x540); adjust or add centering as needed.
#


from pathlib import Path
import subprocess

picked = []
for slide_num, (ok, path, msg) in sorted(saved_files.items()):
    if ok and path:
        picked.append(str(Path(path).resolve()))

print("picked:", picked)
def posix_to_hfs(path):
    script = f'POSIX file "{path}" as text'
    result = subprocess.run(
        ["osascript", "-e", script],
        capture_output=True,
        text=True,
        check=True
    )
    return result.stdout.strip()
picked_hfs = [posix_to_hfs(p) for p in picked]

print(picked_hfs)


import subprocess, textwrap, pathlib 
file_list_string = ", ".join(f'"{p}"' for p in picked_hfs)
applescript = textwrap.dedent(f'''
tell application "Microsoft PowerPoint"
	activate
	set pres to make new presentation	
    delay 1    
    set fileList to {{{file_list_string}}}
    repeat with p in fileList
            delay 1
            set sld to make new slide at end of pres
            set pic to make new picture at sld with properties {{file name:p, left position:0, top:0, lock aspect ratio:false}}
            set width of pic to 960
            set height of pic to 540    

    end repeat
end tell
''').strip() 

subprocess.run(["osascript", "-e", applescript], check=True)

picked: ['/Users/yuetoya/projects/researchOS100-private/notebooks/output/run_20260213_091919/svgs/slide_001_20260213_094104_918.svg', '/Users/yuetoya/projects/researchOS100-private/notebooks/output/run_20260213_091919/svgs/slide_002_20260213_094104_919.svg', '/Users/yuetoya/projects/researchOS100-private/notebooks/output/run_20260213_091919/svgs/slide_003_20260213_094104_920.svg', '/Users/yuetoya/projects/researchOS100-private/notebooks/output/run_20260213_091919/svgs/slide_004_20260213_094104_920.svg', '/Users/yuetoya/projects/researchOS100-private/notebooks/output/run_20260213_091919/svgs/slide_005_20260213_094104_921.svg', '/Users/yuetoya/projects/researchOS100-private/notebooks/output/run_20260213_091919/svgs/slide_006_20260213_094104_921.svg', '/Users/yuetoya/projects/researchOS100-private/notebooks/output/run_20260213_091919/svgs/slide_007_20260213_094104_921.svg']
['Macintosh HD:Users:yuetoya:projects:researchOS100-private:notebooks:output:run_20260213_091919:svgs:slide_001_2026

CompletedProcess(args=['osascript', '-e', 'tell application "Microsoft PowerPoint"\n\tactivate\n\tset pres to make new presentation\t\n    delay 1    \n    set fileList to {"Macintosh HD:Users:yuetoya:projects:researchOS100-private:notebooks:output:run_20260213_091919:svgs:slide_001_20260213_094104_918.svg", "Macintosh HD:Users:yuetoya:projects:researchOS100-private:notebooks:output:run_20260213_091919:svgs:slide_002_20260213_094104_919.svg", "Macintosh HD:Users:yuetoya:projects:researchOS100-private:notebooks:output:run_20260213_091919:svgs:slide_003_20260213_094104_920.svg", "Macintosh HD:Users:yuetoya:projects:researchOS100-private:notebooks:output:run_20260213_091919:svgs:slide_004_20260213_094104_920.svg", "Macintosh HD:Users:yuetoya:projects:researchOS100-private:notebooks:output:run_20260213_091919:svgs:slide_005_20260213_094104_921.svg", "Macintosh HD:Users:yuetoya:projects:researchOS100-private:notebooks:output:run_20260213_091919:svgs:slide_006_20260213_094104_921.svg", "Maci